1- Imports

In [42]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
#Check gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

2- Dataset

In [43]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.SVHN(
    root="./data",
    split="train",
    download=True,
    transform=transform
)
test_dataset = torchvision.datasets.SVHN(
    root="./data",
    split="test",
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

3- CNN Model

In [44]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # Block 1
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2, 2)

        # Block 2
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        # Block 3
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        # Fully connected
        self.fc1 = nn.Linear(64 * 4 * 4, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

4- Training

In [45]:
epochs = 10

for epoch in range(epochs):
    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {running_loss/len(train_loader):.4f} "
          f"Train Acc: {acc:.2f}%")

Epoch [1/10] Loss: 0.9413 Train Acc: 68.58%
Epoch [2/10] Loss: 0.5936 Train Acc: 81.10%
Epoch [3/10] Loss: 0.5189 Train Acc: 83.45%
Epoch [4/10] Loss: 0.4796 Train Acc: 84.69%
Epoch [5/10] Loss: 0.4475 Train Acc: 85.87%
Epoch [6/10] Loss: 0.4161 Train Acc: 87.02%
Epoch [7/10] Loss: 0.3867 Train Acc: 87.81%
Epoch [8/10] Loss: 0.3717 Train Acc: 88.36%
Epoch [9/10] Loss: 0.3475 Train Acc: 89.14%
Epoch [10/10] Loss: 0.3290 Train Acc: 89.87%


5- Testing

In [46]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("\nFinal Test Accuracy: {:.2f}%".format(100 * correct / total))


Final Test Accuracy: 91.36%
